# Invoice Dataset Processor
Run all cells. Results saved to `extracted_invoices.csv` in Google Drive.

In [ ]:
!apt-get install tesseract-ocr tesseract-ocr-fra poppler-utils -y
!pip install transformers torch pillow pytesseract pdf2image pandas tqdm pdfplumber


In [ ]:
from google.colab import drive
drive.mount('/content/drive')


In [ ]:
import os, re, shutil
import torch
import pandas as pd
import pdfplumber
import pytesseract
from PIL import Image
from transformers import pipeline
from pdf2image import convert_from_path
from tqdm import tqdm

# Copy dataset locally for 10x faster I/O
DRIVE_DATASET = '/content/drive/MyDrive/dataset'
LOCAL_DATASET = '/content/dataset'

if os.path.exists(DRIVE_DATASET):
    if not os.path.exists(LOCAL_DATASET):
        print('Copying dataset to local storage for speed...')
        shutil.copytree(DRIVE_DATASET, LOCAL_DATASET)
        print('Done!')
else:
    print(f'Dataset not found at {DRIVE_DATASET}')
    LOCAL_DATASET = DRIVE_DATASET

# Load AI model
device = 0 if torch.cuda.is_available() else -1
print(f"Using device: {'GPU' if device == 0 else 'CPU'}")
try:
    doc_qa = pipeline('document-question-answering', model='impira/layoutlm-document-qa', device=device)
except:
    doc_qa = None


In [ ]:
def clean_amount(val):
    if not val:
        return 0.0
    val = str(val)
    val = val.replace('EUR', '').replace('TND', '').replace(' ', '')
    val = val.replace(',', '.').replace('E', '').replace('e', '')
    val = re.sub(r'[^\d.]', '', val)
    try:
        return float(val) if val else 0.0
    except:
        return 0.0


def process_invoice(file_path):
    text = ''
    is_text_pdf = False

    # Step 1: Try fast text extraction with pdfplumber
    if file_path.lower().endswith('.pdf'):
        try:
            with pdfplumber.open(file_path) as pdf:
                for page in pdf.pages:
                    pt = page.extract_text()
                    if pt:
                        text += pt + '\n'
            if len(text.strip()) > 50:
                is_text_pdf = True
        except:
            pass

    # Step 2: OCR fallback for scanned PDFs or images
    temp_img = None
    img_obj = None
    if not is_text_pdf:
        if file_path.lower().endswith('.pdf'):
            try:
                imgs = convert_from_path(file_path, dpi=150)
                temp_img = file_path + '.jpg'
                imgs[0].save(temp_img, 'JPEG', quality=85)
                img_obj = Image.open(temp_img).convert('RGB')
            except:
                return None
        else:
            try:
                img_obj = Image.open(file_path).convert('RGB')
            except:
                return None
        try:
            text = pytesseract.image_to_string(img_obj, lang='fra', config='--psm 6')
        except:
            text = ''

    # Step 3: Regex extraction
    info = {}
    rx = {
        'invoiceNumber': r'(?:Facture|N.\s*Facture|R.f)[:\s]*([A-Z0-9][A-Z0-9\-/]+)',
        'date': r'(?:Date|Le)[:\s]*(\d{1,2}[-/.]\d{1,2}[-/.]\d{2,4})',
        'totalHT': r'(?:Total\s*H\.?T|Montant\s*HT|Sous.total)[:\s]*([\d\s,.]+)',
        'totalTTC': r'(?:Total\s*T\.?T\.?C|Net\s*.\s*payer|Montant\s*TTC)[:\s]*([\d\s,.]+)',
        'tvaAmount': r'(?:TVA|Taxe|Montant\s*TVA)[:\s]*([\d\s,.]+)',
        'timbre': r'(?:Timbre|Droit de timbre)[:\s]*([\d\s,.]+)',
        'client': r'(?:Client|Factur.\s*.)[:\s]*([A-Za-z\u00c0-\u00ff\s]{3,})',
        'companyName': r'([A-Za-z\u00c0-\u00ff0-9\s]{3,}(?:SARL|SAS|SA|EURL))',
    }
    for key, pattern in rx.items():
        m = re.search(pattern, text, re.IGNORECASE)
        if m:
            info[key] = m.group(1).strip()

    # Step 4: AI fallback for missing totalTTC
    if doc_qa and img_obj and 'totalTTC' not in info:
        try:
            ans = doc_qa(img_obj, 'What is the total TTC amount?')
            if ans:
                info['totalTTC'] = ans[0]['answer']
        except:
            pass

    if temp_img and os.path.exists(temp_img):
        os.remove(temp_img)

    ht = clean_amount(info.get('totalHT'))
    tva = clean_amount(info.get('tvaAmount'))
    timbre = clean_amount(info.get('timbre'))
    ttc = clean_amount(info.get('totalTTC'))

    return {
        'companyName': info.get('companyName', ''),
        'invoiceNumber': info.get('invoiceNumber', ''),
        'date': info.get('date', ''),
        'client': info.get('client', ''),
        'totalHT': ht,
        'tvaAmount': tva,
        'timbre': timbre,
        'totalTTC': ttc,
        'rawText': text[:600].replace(chr(10), ' ') if text else '',
    }


In [ ]:
# Collect all files
files = []
for root, dirs, fnames in os.walk(LOCAL_DATASET):
    for f in fnames:
        if f.lower().endswith(('.pdf', '.jpg', '.jpeg', '.png')):
            files.append(os.path.join(root, f))

print(f'Found {len(files)} files.')

OUTPUT_CSV = '/content/drive/MyDrive/extracted_invoices.csv'
results = []

for fp in tqdm(files, desc='Processing'):
    r = process_invoice(fp)
    if r:
        r['filename'] = os.path.basename(fp)
        results.append(r)
        # Save after each file so nothing is lost
        pd.DataFrame(results).to_csv(OUTPUT_CSV, index=False)

print(f'Done! {len(results)} invoices saved to Google Drive.')
